# Boston House Price Prediction — EDA and Modeling

This notebook is a readable walkthrough of the same modular pipeline used by `src/train.py`. It inspects the supplied dataset, validates it, explores the data, compares regressors, and summarizes the tuned final model.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_preprocessing import load_dataset, identify_index_columns, infer_target_column, data_quality_report, prepare_features

DATA_PATH = ROOT / "data" / "BostonHousing.csv"
df = load_dataset(DATA_PATH)
index_columns = identify_index_columns(df)
target_column = infer_target_column(df, index_columns)

print("Shape:", df.shape)
print("Target:", target_column)
print("Index-like columns:", index_columns)
df.head()

In [ ]:
# Data quality checks
quality = data_quality_report(df, target_column, index_columns)
print("Columns:", quality["column_names"])
print("Dtypes:")
print(pd.Series(quality["dtypes"]))
print("Missing values:")
print(pd.Series(quality["missing_values"]))
print("Duplicate rows:", quality["duplicate_rows"])
print("IQR outlier counts:")
print(pd.Series(quality["outlier_counts_iqr"]))

In [ ]:
# Statistical summary
print(df.describe().T)

In [ ]:
# EDA figures used by the project
from src.evaluate import save_eda_plots
save_eda_plots(df.drop(columns=index_columns, errors="ignore"), target_column, ROOT / "outputs" / "figures")
print("EDA figures saved to outputs/figures/")

## Modeling approach

The data is split once into training and test sets. Feature imputation, scaling, and encoding live inside sklearn pipelines, so those transformations are learned only from the relevant training folds. Model selection uses mean 5-fold cross-validation RMSE on the training data.

In [ ]:
from IPython.display import display
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from src.data_preprocessing import build_preprocessor
from src.evaluate import regression_metrics

X, y = prepare_features(df, target_column, index_columns)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
preprocessor, numerical_columns, categorical_columns = build_preprocessor(X_train)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

rows = []
for name, estimator in models.items():
    pipe = Pipeline([("preprocessing", preprocessor), ("model", estimator)])
    cv = cross_validate(pipe, X_train, y_train, cv=5,
                        scoring={"rmse":"neg_root_mean_squared_error", "mae":"neg_mean_absolute_error", "r2":"r2"},
                        n_jobs=-1)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    test = regression_metrics(y_test, pred)
    rows.append({"Model": name,
                 "CV RMSE": -cv["test_rmse"].mean(),
                 "CV MAE": -cv["test_mae"].mean(),
                 "CV R2": cv["test_r2"].mean(),
                 **{f"Test {k}": v for k, v in test.items()}})

comparison = pd.DataFrame(rows).sort_values("CV RMSE")
comparison

In [ ]:
# Tune the model selected by CV
from sklearn.model_selection import GridSearchCV

best_name = comparison.iloc[0]["Model"]
print("Selected baseline:", best_name)

selected = models[best_name]
pipe = Pipeline([("preprocessing", preprocessor), ("model", selected)])

if best_name == "Gradient Boosting":
    grid = {
        "model__n_estimators": [100, 200, 300],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3],
        "model__min_samples_leaf": [1, 3],
    }
elif best_name == "Random Forest":
    grid = {
        "model__n_estimators": [300, 500],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": [1.0, "sqrt"],
    }
elif best_name == "Ridge Regression":
    grid = {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]}
else:
    grid = {}

if grid:
    search = GridSearchCV(pipe, grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
    search.fit(X_train, y_train)
    final_model = search.best_estimator_
    print("Best parameters:", search.best_params_)
    print("Best CV RMSE:", -search.best_score_)
else:
    final_model = pipe.fit(X_train, y_train)

final_pred = final_model.predict(X_test)
final_metrics = regression_metrics(y_test, final_pred)
print("Final test metrics:", final_metrics)

In [ ]:
# Final diagnostics and feature importance
from src.evaluate import save_prediction_plots, save_feature_importance
save_prediction_plots(y_test, final_pred, ROOT / "outputs" / "figures")

names = list(final_model.named_steps["preprocessing"].get_feature_names_out())
estimator = final_model.named_steps["model"]
if hasattr(estimator, "feature_importances_"):
    importance = save_feature_importance(names, estimator.feature_importances_, ROOT / "outputs" / "figures")
    display(importance.head(15))
else:
    print("Final estimator does not expose tree feature_importances_.")

## Production handoff

The notebook is intentionally a walkthrough. For repeatable training and deployment, run `python -m src.train` from the project root. That script saves the complete fitted pipeline to `models/house_price_model.pkl`, a separate preprocessing artifact, metadata, comparison metrics, and diagnostic figures.